# 🔬 Leukemia Detection using VGG16 Transfer Learning

Binary classification of blood cell images into **Healthy (HEM)** vs **Leukemia (ALL)** using a pre-trained VGG16 model.

**Dataset:** CNMC Leukemia (C-NMC Challenge)  
**Model:** VGG16 (ImageNet weights, frozen base)  
**Image Size:** 224×224  
**Approach:** Transfer Learning — no data augmentation

---
## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

---
## 2. Configuration

In [ ]:
# ── Hyperparameters & paths ──────────────────────────────────────────
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 10
LEARNING_RATE = 1e-4
SEED         = 42

BASE_DIR = '/kaggle/input/leukemia/'

# Class mapping
CLASS_NAMES = ['all', 'hem']   # alphabetical order used by flow_from_directory
# all = Leukemia (positive), hem = Healthy (negative)

---
## 3. Dataset Loading

In [ ]:
# ── Explore the dataset structure ────────────────────────────────────
for item in sorted(os.listdir(BASE_DIR)):
    item_path = os.path.join(BASE_DIR, item)
    if os.path.isdir(item_path):
        sub = os.listdir(item_path)
        print(f"📁 {item}/  →  {sub}")

In [ ]:
import glob

# ── Collect all image paths and labels from fold_0, fold_1, fold_2 ──
all_images = []
all_labels = []

fold_dirs = sorted(glob.glob(os.path.join(BASE_DIR, 'fold_*')))
print(f"Found {len(fold_dirs)} folds: {[os.path.basename(f) for f in fold_dirs]}\n")

for fold in fold_dirs:
    for cls in ['all', 'hem']:
        cls_dir = os.path.join(fold, cls)
        if os.path.isdir(cls_dir):
            imgs = os.listdir(cls_dir)
            print(f"  {os.path.basename(fold)}/{cls}: {len(imgs)} images")
            for img in imgs:
                all_images.append(os.path.join(cls_dir, img))
                all_labels.append(cls)

print(f"\nTotal images collected: {len(all_images)}")

---
## 4. Data Preprocessing

In [ ]:
# ── Use fold_0 and fold_1 for training, fold_2 for validation ───────
train_dirs = [d for d in fold_dirs if 'fold_2' not in d]
val_dirs   = [d for d in fold_dirs if 'fold_2' in d]

print(f"Training folds : {[os.path.basename(d) for d in train_dirs]}")
print(f"Validation fold: {[os.path.basename(d) for d in val_dirs]}")

In [ ]:
import shutil, tempfile, pathlib

# ── Create temporary merged directories for ImageDataGenerator ──────
# We merge fold_0 + fold_1 → train_dir and fold_2 → val_dir
# using symlinks so we don't duplicate data.

tmp_root  = '/tmp/leukemia_split'
train_dir = os.path.join(tmp_root, 'train')
val_dir   = os.path.join(tmp_root, 'val')

def link_images(src_folds, dst_root):
    """Create symlinks from source fold class dirs into a merged directory."""
    for cls in ['all', 'hem']:
        cls_dst = os.path.join(dst_root, cls)
        os.makedirs(cls_dst, exist_ok=True)
        for fold in src_folds:
            cls_src = os.path.join(fold, cls)
            if not os.path.isdir(cls_src):
                continue
            for fname in os.listdir(cls_src):
                src = os.path.join(cls_src, fname)
                dst = os.path.join(cls_dst, fname)
                if not os.path.exists(dst):
                    os.symlink(src, dst)

# Clean previous run (if any)
if os.path.exists(tmp_root):
    shutil.rmtree(tmp_root)

link_images(train_dirs, train_dir)
link_images(val_dirs,   val_dir)

# Verify counts
for split, d in [('Train', train_dir), ('Val', val_dir)]:
    for cls in ['all', 'hem']:
        n = len(os.listdir(os.path.join(d, cls)))
        print(f"  {split} / {cls}: {n} images")

In [ ]:
# ── ImageDataGenerators (rescale only, NO augmentation) ─────────────
train_datagen = ImageDataGenerator(rescale=1.0 / 255)
val_datagen   = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nClass indices: {train_generator.class_indices}")

In [ ]:
# ── Visualise a batch of training images ─────────────────────────────
label_map = {v: k for k, v in train_generator.class_indices.items()}

batch_imgs, batch_labels = next(train_generator)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(batch_imgs[i])
    cls_name = label_map[int(batch_labels[i])]
    color = 'red' if cls_name == 'all' else 'green'
    ax.set_title(cls_name.upper(), fontsize=12, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Model Building

In [ ]:
# ── Load VGG16 pre-trained on ImageNet (exclude top layers) ─────────
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all base model layers
base_model.trainable = False

print(f"Base model layers: {len(base_model.layers)}")
print(f"Trainable weights: {len(base_model.trainable_weights)}")

In [ ]:
# ── Add custom classification head ─────────────────────────────────
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)   # binary classification

model = Model(inputs=base_model.input, outputs=output)

print(f"Total params: {model.count_params():,}")
print(f"Trainable params: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")
print(f"Non-trainable params: {sum(tf.keras.backend.count_params(w) for w in model.non_trainable_weights):,}")

---
## 6. Model Summary

In [ ]:
model.summary()

---
## 7. Model Compilation

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled successfully.")

---
## 8. Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

---
## 9. Model Training

In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

---
## 10. Results Visualization

In [ ]:
# ── Training & Validation — Accuracy and Loss curves ────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'],  label='Val Accuracy',   linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'],     label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'],  label='Val Loss',   linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Evaluation

In [ ]:
# ── Evaluate on the validation set ─────────────────────────────────
val_loss, val_acc = model.evaluate(val_generator, verbose=1)
print(f"\n📊 Validation Loss:     {val_loss:.4f}")
print(f"📊 Validation Accuracy: {val_acc:.4f}  ({val_acc*100:.2f}%)")

In [ ]:
# ── Generate predictions ───────────────────────────────────────────
val_generator.reset()

y_pred_prob = model.predict(val_generator, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = val_generator.classes

print(f"Predictions shape: {y_pred.shape}")
print(f"True labels shape: {y_true.shape}")

---
## 12. Confusion Matrix

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['ALL (Leukemia)', 'HEM (Healthy)'],
    yticklabels=['ALL (Leukemia)', 'HEM (Healthy)'],
    annot_kws={'size': 14}
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

---
## 13. Classification Report

In [ ]:
# ── Detailed classification metrics ─────────────────────────────────
target_names = ['ALL (Leukemia)', 'HEM (Healthy)']
report = classification_report(y_true, y_pred, target_names=target_names)
print(report)

In [ ]:
# ── Per-class accuracy bar chart ────────────────────────────────────
from sklearn.metrics import classification_report as cr_dict

report_dict = cr_dict(y_true, y_pred, target_names=target_names, output_dict=True)

classes   = target_names
precision = [report_dict[c]['precision'] for c in classes]
recall    = [report_dict[c]['recall']    for c in classes]
f1        = [report_dict[c]['f1-score']  for c in classes]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, precision, width, label='Precision', color='#3498db')
ax.bar(x,         recall,    width, label='Recall',    color='#2ecc71')
ax.bar(x + width, f1,        width, label='F1-Score',  color='#e74c3c')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Metrics per Class', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 14. Sample Predictions

In [ ]:
# ── Show sample predictions on validation images ───────────────────
val_generator.reset()
batch_imgs, batch_labels = next(val_generator)
preds = model.predict(batch_imgs, verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(batch_imgs[i])
    true_cls  = 'ALL' if batch_labels[i] == 0 else 'HEM'
    pred_cls  = 'ALL' if preds[i][0] < 0.5 else 'HEM'
    conf      = preds[i][0] if preds[i][0] > 0.5 else 1 - preds[i][0]
    color     = 'green' if true_cls == pred_cls else 'red'
    ax.set_title(f"True: {true_cls}\nPred: {pred_cls} ({conf:.1%})",
                 fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Validation Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 15. Summary

| Item | Detail |
|---|---|
| **Model** | VGG16 (ImageNet, frozen base) |
| **Classification** | Binary — ALL (Leukemia) vs HEM (Healthy) |
| **Image Size** | 224 × 224 |
| **Augmentation** | None |
| **Optimizer** | Adam (lr=1e-4) |
| **Loss** | Binary Cross-Entropy |
| **Callbacks** | EarlyStopping, ReduceLROnPlateau |